# Análise de Evasão Escolar na Grande Vitória (ES)

Este projeto tem como objetivo analisar os fatores associados à evasão escolar no Ensino Médio na região da Grande Vitória (Vitória, Cariacica, Vila Velha e Serra), utilizando dados do INEP.

A proposta é identificar padrões e relações entre desempenho acadêmico, tipo de rede de ensino e taxas de abandono escolar, buscando compreender quais variáveis estão mais associadas à evasão.

Nesta MVP serão apresentadas as seguintes etapas:

- Análise exploratória dos dados (EDA), incluindo comparações entre níveis de ensino, municípios e tipos de escola
- Análise estatística, incluindo correlação de Pearson para investigação de relações entre variáveis
- Modelagem supervisionada, com Regressão Linear Múltipla para interpretação dos fatores associados à evasão
- Modelagem preditiva com Random Forest Regressor para captura de relações não lineares
- Avaliação e comparação de desempenho dos modelos por meio de métricas como R², MAE e RMSE


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## Carregamento e Filtro Geográfico


In [14]:
cols_micro = ['CO_ENTIDADE', 'NO_MUNICIPIO', 'SG_UF', 'TP_DEPENDENCIA']
cidades = ['Vitória', 'Cariacica', 'Vila Velha', 'Serra']

try:
   df_micro = pd.read_csv('microdados_ed_basica_2024.csv', sep=';', encoding='latin1', usecols=cols_micro )
   df_es = df_micro[(df_micro['SG_UF'] == 'ES') & (df_micro['NO_MUNICIPIO'].isin(cidades))].copy()

   df_tx = pd.read_excel('tx_rend_escolas_2024.xlsx', skiprows=8)
   df_tx.columns = [c.strip() for c in df_tx.columns]

   df_tx['CO_ENTIDADE'] = df_tx['CO_ENTIDADE'].astype(str).str.split('.').str[0]
   df_es['CO_ENTIDADE'] = df_es['CO_ENTIDADE'].astype(str).str.split('.').str[0]

   df_final = pd.merge(df_es, df_tx, on='CO_ENTIDADE', how='inner')
        
   if df_final.empty:
           print("Atenção: O merge resultou em um dataframe vazio. Verifique os códigos das escolas.")
   else:
           print(f"Sucesso! {len(df_final)} escolas da Grande Vitória encontradas.")

except Exception as e:
    print(f"Erro ao processar: {e}")

Erro ao processar: [Errno 2] No such file or directory: 'microdados_ed_basica_2024.csv'


## Conjunto de Dados Macro

In [15]:
print(df_final.head())

NameError: name 'df_final' is not defined

In [16]:
df_final.shape

NameError: name 'df_final' is not defined

In [17]:
print(df_final.info())

NameError: name 'df_final' is not defined

## Verificação de dados nulos

In [18]:
print(df_final.isnull().sum())

NameError: name 'df_final' is not defined

## Calculando as taxas de Abandono por Nível de Ensino

In [19]:
cols_abandono_fund = [f'3_CAT_FUN_{i:02d}' for i in range(1, 10)]
cols_abandono_med = [f'3_CAT_MED_{i:02d}' for i in range(1, 4)]

df_final[cols_abandono_fund] = df_final[cols_abandono_fund].apply(
    pd.to_numeric,
    errors='coerce'
)

df_final[cols_abandono_med] = df_final[cols_abandono_med].apply(
    pd.to_numeric,
    errors='coerce'
)

NameError: name 'df_final' is not defined

In [20]:
media_fund = df_final[cols_abandono_fund].stack().mean()
media_med = df_final[cols_abandono_med].stack().mean()

print(f"Média de abandono do Fundamental: {media_fund:.2f}")
print(f"Média de abandono do Médio: {media_med:.2f}")

NameError: name 'df_final' is not defined

## Correlação do Abandono com o Município

In [21]:
cols_abandono = cols_abandono_fund + cols_abandono_med

df_final['taxa_abandono_total'] = df_final[cols_abandono].mean(axis=1)

NameError: name 'df_final' is not defined

In [22]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_final, 
            x='NO_MUNICIPIO_x', 
            y='taxa_abandono_total', 
            hue='NO_MUNICIPIO_x', 
            palette="Set2", 
            legend=False          
)

plt.title('Taxa de Abandono no Ensino Médio - Grande Vitória (2019)', fontsize=14)
plt.ylabel('Taxa de Abandono (%)')
plt.xlabel('Município')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

NameError: name 'df_final' is not defined

<Figure size 1200x600 with 0 Axes>

### Comparação da Taxa de Abandono: Fundamental x Médio

In [23]:
df_final['taxa_abandono_fund'] = df_final[cols_abandono_fund].mean(axis=1)
df_final['taxa_abandono_med'] = df_final[cols_abandono_med].mean(axis=1)

comparativo_medias = df_final.groupby('NO_MUNICIPIO_x')[['taxa_abandono_fund', 'taxa_abandono_med']].mean()
print("Média de Abandono por Etapa e Município:")
display(comparativo_medias)

NameError: name 'df_final' is not defined

In [24]:
df_melted = df_final.melt(id_vars=['NO_MUNICIPIO_x'], 
                         value_vars=['taxa_abandono_fund', 'taxa_abandono_med'],
                         var_name='Etapa', value_name='Taxa_Abandono')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='NO_MUNICIPIO_x', y='Taxa_Abandono', hue='Etapa', palette='muted')

plt.title('Comparação da Taxa de Abandono: Fundamental vs Médio', fontsize=14)
plt.ylabel('Taxa Média de Abandono (%)')
plt.xlabel('Município')
plt.legend(title='Etapa de Ensino')
plt.show()

NameError: name 'df_final' is not defined

No geral, a taxa média de abandono no ensino médio (1,13%) é aproximadamente 5,4 vezes maior que a observada no ensino fundamental (0,21%).

Isso pode estar associado a fatores como:
- entrada precoce no mercado de trabalho
- desmotivação com o modelo educacional
- aumento da dificuldade acadêmica

O Ensino Médio deve ser tratado como etapa crítica para políticas de permanência escolar.

## Fatores de Desempenho Escolar x Evasão

In [25]:
cols_aprovacao_fund = [f'1_CAT_FUN_{i:02d}' for i in range(1, 10)]
cols_aprovacao_med = [f'1_CAT_MED_{i:02d}' for i in range(1, 4)]

df_final[cols_aprovacao_fund] = df_final[cols_aprovacao_fund].apply(
    pd.to_numeric,
    errors='coerce'
)

df_final[cols_aprovacao_med] = df_final[cols_aprovacao_med].apply(
    pd.to_numeric,
    errors='coerce'
)

cols_aprovacao = cols_aprovacao_fund + cols_aprovacao_med

df_final['taxa_aprovacao_total'] = df_final[cols_aprovacao].mean(axis=1)

NameError: name 'df_final' is not defined

### Correlação de Pearson entre a Taxa de Aprovação e a Taxa de Abandono

In [26]:
df_final[['taxa_aprovacao_total', 'taxa_abandono_total']].corr()

NameError: name 'df_final' is not defined

In [27]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_final, x='taxa_aprovacao_total', y='taxa_abandono_total', hue='NO_MUNICIPIO_x', s=100, alpha=0.7)
plt.title('Correlação: Taxa de Aprovação vs Taxa de Abandono')
plt.xlabel('Taxa de Aprovação (%)')
plt.ylabel('Taxa de Abandono (%)')
plt.axhline(df_final['taxa_abandono_total'].mean(), color='red', linestyle='--', label='Média de Abandono')
plt.legend()
plt.show()

NameError: name 'df_final' is not defined

<Figure size 1000x600 with 0 Axes>

Existe uma correlação negativa moderada entre aprovação e evasão. Em geral, municípios com maiores taxas de aprovação tendem a apresentar menores taxas de abandono escolar,  indicando que dificuldades de aprendizagem podem desestimular os alunos, consequentemente, à maior probabilidade de evasão escolar.

## Proporção de Abandono por tipo de escola 

In [28]:
dict_dep = {1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'}
df_final['Rede'] = df_final['TP_DEPENDENCIA'].map(dict_dep)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_final, x='Rede', y='taxa_abandono_total', hue='Rede', palette='coolwarm', legend=False)
plt.title('Taxa de Abandono no Ensino Médio por Rede de Ensino')
plt.ylabel('Taxa de Abandono (%)')
plt.show()

NameError: name 'df_final' is not defined

Escolas públicas apresentam maior variabilidade e, em geral, maiores taxas de abandono em comparação às escolas privadas.

Isso pode refletir desigualdades socioeconômicas, onde alunos da rede pública enfrentam maiores desafios, como:
- necessidade de trabalhar
- menor apoio familiar
- menor acesso a recursos educacionais

## Regressão Linear Simples

In [29]:
from sklearn.linear_model import LinearRegression

dados = df_final[['taxa_aprovacao_total', 'taxa_abandono_total']].dropna()

X = dados[['taxa_aprovacao_total']]
y = dados['taxa_abandono_total']

modelo = LinearRegression()
modelo.fit(X, y)

print("Intercepto:", modelo.intercept_)
print("Coeficiente:", modelo.coef_[0])

NameError: name 'df_final' is not defined

A regressão linear simples indicou uma relação negativa entre a taxa de aprovação e a taxa de abandono escolar. O coeficiente estimado (-0,131) sugere que aumentos na taxa de aprovação estão associados à redução da evasão escolar. Em termos práticos, para cada aumento de 1 ponto percentual na taxa de aprovação, espera-se uma redução média de aproximadamente 0,13 pontos percentuais na taxa de abandono. Esse resultado reforça os achados da análise de correlação, que apontou uma associação negativa moderada entre as variáveis.

In [30]:
from sklearn.metrics import r2_score

y_pred = modelo.predict(X)

print("R²:", r2_score(y, y_pred))

NameError: name 'modelo' is not defined

In [31]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=df_final,
    x='taxa_aprovacao_total',
    y='taxa_abandono_total',
    scatter_kws={'alpha': 0.6},
    line_kws={'linewidth': 2}
)

plt.title('Relação entre Taxa de Aprovação e Taxa de Abandono')
plt.xlabel('Taxa de Aprovação (%)')
plt.ylabel('Taxa de Abandono (%)')

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

NameError: name 'df_final' is not defined

<Figure size 1000x600 with 0 Axes>

A regressão linear simples apresentou R² = 0,17, indicando que aproximadamente 17% da variação da taxa de abandono pode ser explicada pela taxa de aprovação. Embora a relação seja estatisticamente relevante, o resultado sugere que outros fatores também exercem influência significativa sobre a evasão escolar.

## Regressão Linear Múltipla

In [ ]:
df_modelo = pd.get_dummies(
    df_final,
    columns=['TP_DEPENDENCIA'],
    drop_first=True
)

In [ ]:
features = [
    'taxa_aprovacao_total',
    'TP_DEPENDENCIA_2',
    'TP_DEPENDENCIA_3',
    'TP_DEPENDENCIA_4'
]

target = 'taxa_abandono_total'

dados = df_modelo[features + [target]].dropna()

X = dados[features]
y = dados[target]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_train, y_train)

In [ ]:
y_pred = modelo.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

In [ ]:
coeficientes = pd.DataFrame({
    'Variavel': X.columns,
    'Coeficiente': modelo.coef_
})

coeficientes.sort_values(
    by='Coeficiente',
    ascending=False
)

A regressão linear múltipla foi utilizada para investigar a influência da taxa de aprovação e da dependência administrativa sobre a evasão escolar. Embora os coeficientes indiquem associações coerentes com a literatura, o modelo apresentou baixo poder explicativo (R² ≈ 0), sugerindo que a evasão escolar é influenciada por fatores adicionais não contemplados nesta análise.

## Random Forest

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

if 'const' in X.columns:
    X = X.drop(columns='const')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

modelo_rf.fit(X_train, y_train)

In [ ]:
y_pred = modelo_rf.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
importancias = pd.DataFrame({
    'variavel': X.columns,
    'importancia': modelo_rf.feature_importances_
}).sort_values(by='importancia', ascending=False)

importancias

In [ ]:
plt.bar(importancias['variavel'], importancias['importancia'])
plt.xticks(rotation=45)
plt.title("Importância das Variáveis - Random Forest")
plt.show()

## Conclusão

Embora tenha sido observada uma associação estatisticamente significativa entre aprovação e evasão escolar, os modelos preditivos apresentaram baixo desempenho, sugerindo que a evasão é influenciada por fatores adicionais não contemplados na base analisada. Isso evidencia a complexidade do fenômeno e a necessidade de incorporar variáveis socioeconômicas e educacionais complementares em estudos futuros.